# CASCADE-ER v2 — Stage 1: WIDE BLOCKING Runner (Google Colab)

This notebook executes **Stage 1 (Wide Blocking)** of the CASCADE-ER v2 entity-resolution pipeline on Google Colab with GPU acceleration and Google Drive integration.

### Stage 1 Purpose & Objectives:
* **MAXIMIZE CANDIDATE RECALL** across 6 complementary blocking channels.
* **Multi-channel Union**:
  1. Exact Normalized Name Block
  2. Core Name Block (corporate legal suffix stripping)
  3. House Number + Street Token Block
  4. Postal / Zip Code Block
  5. Character-level TF-IDF Retrieval (Top-K=30)
  6. Multilingual Dense Embeddings & FAISS Retrieval (Top-K=20)
* **Output**: Internal deduplicated candidate union (`data/candidates/stage1_candidate_union.tsv`).
* **Recall Evaluation**: Automated recall scoring against `train_ground_truth.tsv` (overall union recall & per-channel recall).

--- 
## 1. Clone / Update Repository & Install Dependencies

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/Blackdevil-com/Wired-Mortus---Amazon-ML.git"
REPO_NAME = "Wired-Mortus---Amazon-ML"

# Clone if not present, then enter directory and pull latest code
if not Path("src/blocking").exists():
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    %cd {REPO_NAME}

!git fetch origin main
!git reset --hard origin/main

print(f"\nCurrent working directory: {Path.cwd()}")
!pip install -r requirements-colab.txt

--- 
## 2. Verify GPU & Environment Setup

In [ ]:
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")

--- 
## 3. Mount Google Drive & Link Datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = Path('/content/drive/MyDrive/entity-resolution')
DRIVE_RAW_TRAIN = DRIVE_BASE / 'data/raw/train'
DRIVE_PROC_TRAIN = DRIVE_BASE / 'data/processed/train'
DRIVE_CANDIDATES = DRIVE_BASE / 'data/candidates'

!mkdir -p data/raw/train data/processed/train data/candidates data/cache
DRIVE_CANDIDATES.mkdir(parents=True, exist_ok=True)

# Copy preprocessed datasets or raw datasets if available in Drive
if DRIVE_PROC_TRAIN.exists():
    print(f"Copying preprocessed datasets from {DRIVE_PROC_TRAIN}...")
    !cp -n "{DRIVE_PROC_TRAIN}"/*.tsv data/processed/train/ 2>/dev/null || true

if DRIVE_RAW_TRAIN.exists():
    !cp -n "{DRIVE_RAW_TRAIN}"/train_ground_truth.tsv data/raw/train/ 2>/dev/null || true

print("\n--- Local Datasets Status ---")
for f in sorted(Path("data/processed/train").glob("*.tsv")):
    print(f"  - {f.name} ({f.stat().st_size / (1024 * 1024):.2f} MB)")

--- 
## 4. Run Pytest Suite

In [ ]:
!pytest tests/ -v

--- 
## 5. Execute Stage 1: Wide Blocking (High-Recall Candidate Generation)

In [ ]:
# Execute Stage 1 Wide Blocking pipeline
!python run_stage1_blocking.py \
    --split train \
    --s1 data/processed/train/train_source1_preprocessed.tsv \
    --s2 data/processed/train/train_source2_preprocessed.tsv \
    --s3 data/processed/train/train_source3_preprocessed.tsv \
    --ground-truth data/raw/train/train_ground_truth.tsv \
    --tfidf-top-k 30 \
    --embedding-top-k 20 \
    --output data/candidates/stage1_candidate_union.tsv

# Copy candidate union to Google Drive
print("\nUploading candidate union to Google Drive...")
!cp -v data/candidates/stage1_candidate_union.tsv "{DRIVE_CANDIDATES}/"

--- 
## 6. Inspect Candidate Union Sample Records

In [ ]:
import pandas as pd

cand_file = Path("data/candidates/stage1_candidate_union.tsv")
if cand_file.exists():
    df = pd.read_csv(cand_file, sep="\t", nrows=10, dtype=str, keep_default_na=False)
    print(f"Candidate Union Preview (Total columns: {len(df.columns)}):\n")
    display(df)